In [4]:
from PIL import Image
import numpy as np

# Create a dummy image for 'sample.jpg'
# This creates a black 224x224 RGB image
dummy_image = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
dummy_image.save('sample.jpg')

print('Generated dummy sample.jpg')

Generated dummy sample.jpg


In [8]:
import os
import shutil
from PIL import Image
import numpy as np

# Define paths
base_dir = 'dataset'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# Clean up existing directories if they exist
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

# Create base directory
os.makedirs(base_dir, exist_ok=True)

# Create train, validation, test directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(validation_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Create class subdirectories (e.g., 'class_a', 'class_b')
classes = ['class_a', 'class_b']
for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(validation_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

# Create dummy images
def create_dummy_image(path, filename, size=(224, 224)):
    dummy_image_array = np.random.randint(0, 255, size=(size[0], size[1], 3), dtype=np.uint8)
    img = Image.fromarray(dummy_image_array)
    img.save(os.path.join(path, filename))

# Populate with dummy images
num_images_per_class_train = 5
num_images_per_class_val = 2
num_images_per_class_test = 2

print("Creating dummy dataset structure...")
for cls in classes:
    for i in range(num_images_per_class_train):
        create_dummy_image(os.path.join(train_dir, cls), f'img_{i}.jpg')
    for i in range(num_images_per_class_val):
        create_dummy_image(os.path.join(validation_dir, cls), f'img_{i}.jpg')
    for i in range(num_images_per_class_test):
        create_dummy_image(os.path.join(test_dir, cls), f'img_{i}.jpg')

print("Dummy dataset created successfully.")

Creating dummy dataset structure...
Dummy dataset created successfully.


In [5]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np

# Load pretrained model
model = MobileNet(weights='imagenet')

# Load image
img_path = 'sample.jpg'
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# Predict
preds = model.predict(img_array)

# Show top predictions
print(decode_predictions(preds, top=3)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 829ms/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[('n03729826', 'matchstick', np.float32(0.052282106)), ('n04286575', 'spotlight', np.float32(0.047154408)), ('n03196217', 'digital_clock', np.float32(0.044907298))]


In [6]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = ResNet50(weights='imagenet',
                      include_top=False,
                      input_shape=(224,224,3))

# Freeze layers
for layer in base_model.layers[:-10]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input,
              outputs=predictions)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,242 (90.98 MB)

 Trainable params: 4,728,194 (18.04 MB)

 Non-trainable params: 19,122,048 (72.94 MB)

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

train_datagen = ImageDataGenerator(
    rescale=1./255
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

# For test data, usually no augmentation and only rescaling
test_datagen = ImageDataGenerator(rescale=1./255)

print(f"Processing training data from: {train_dir}")
train_data = train_datagen.flow_from_directory(
    train_dir, # Use the specific train_dir from the previously generated cell
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)
print(f"Train data classes found: {train_data.num_classes}, class indices: {train_data.class_indices}")

print(f"Processing validation data from: {validation_dir}")
val_data = val_datagen.flow_from_directory(
    validation_dir, # Use the specific validation_dir from the previously generated cell
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)
print(f"Validation data classes found: {val_data.num_classes}, class indices: {val_data.class_indices}")

# Define test_data
print(f"Processing test data from: {test_dir}")
test_data = test_datagen.flow_from_directory(
    test_dir, # Use the specific test_dir from the previously generated cell
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False # Typically shuffle=False for test data to maintain order for evaluation
)
print(f"Test data classes found: {test_data.num_classes}, class indices: {test_data.class_indices}")


base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
output = Dense(train_data.num_classes,
               activation='softmax')(x)

model = Model(base_model.input, output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10 # Reverted epochs to 10
)

Processing training data from: dataset/train
Found 10 images belonging to 2 classes.
Train data classes found: 2, class indices: {'class_a': 0, 'class_b': 1}
Processing validation data from: dataset/validation
Found 4 images belonging to 2 classes.
Validation data classes found: 2, class indices: {'class_a': 0, 'class_b': 1}
Processing test data from: dataset/test
Found 4 images belonging to 2 classes.
Test data classes found: 2, class indices: {'class_a': 0, 'class_b': 1}
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.5000 - loss: 0.7297 - val_accuracy: 0.5000 - val_loss: 0.7016
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 576ms/step - accuracy: 0.5000 - loss: 0.7236 - val_accuracy: 0.5000 - val_loss: 0.6948
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 760ms/step - accuracy: 0.5000 - loss: 0.7067 - val_accuracy: 0.5000 - val_loss: 0.6647
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6000 - loss: 0.6729 - val_accuracy: 0.5000 - val_loss: 0.6675
Epoch 5/10
1/1 ━━━━━━━━━

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Predict test set
predictions = model.predict(test_data)

y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes

# Classification report
print(classification_report(y_true, y_pred))

# Confusion matrix
print(confusion_matrix(y_true, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      0.50      0.40         2

    accuracy                           0.25         4
   macro avg       0.17      0.25      0.20         4
weighted avg       0.17      0.25      0.20         4

[[0 2]
 [1 1]]
